# Бенчмарк CSR-N vs pydata/sparse vs scipy.sparse

Сравнение CSR-N с библиотечными бейзлайнами на сетке из 13 конфигурациях (N = 2 … 8, плотности 10⁻⁴ … 10⁻²). Замеры построения, TTM по моде 0, случайный доступ, память + отдельные секции для TTV, TTM с разреженной матрицей и TTT.

In [ ]:
!pip install -q sparse tabulate

import os, gc, sys, time
import numpy as np
import pandas as pd
import scipy.sparse as sp
import sparse
from tabulate import tabulate

from csrn import CSRN

os.makedirs('results', exist_ok=True)
REPORT_LINES = []

print(f'numpy   {np.__version__}')
print(f'scipy   {getattr(sp, "__version__", "n/a")}')
print(f'sparse  {sparse.__version__}')
print(f'pandas  {pd.__version__}')
print(f'Python  {sys.version.split()[0]}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.9/151.9 kB 3.9 MB/s eta 0:00:00
numpy   2.0.2
scipy   n/a
sparse  0.18.0
pandas  2.2.2
Python  3.12.13


## Утилиты

In [ ]:
def timeit(fn, n_repeat=3, warmup=1):
    """Медиана n_repeat замеров"""
    for _ in range(warmup):
        fn()
    times = []
    for _ in range(n_repeat):
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    return float(np.median(times))


def fmt_time(t):
    if t is None or (isinstance(t, float) and np.isnan(t)):
        return '—'
    if t < 1e-3: return f'{t*1e6:.1f} µs'
    if t < 1:    return f'{t*1e3:.2f} ms'
    return f'{t:.3f} s'


def fmt_bytes(b):
    if b is None or (isinstance(b, float) and np.isnan(b)):
        return '—'
    b = float(b)
    for unit in ['B', 'KB', 'MB']:
        if b < 1024: return f'{b:.1f} {unit}'
        b /= 1024
    return f'{b:.1f} GB'


def make_random_tensor(shape, density, seed=0):
    """Случайный разреженный тензор, N(0,1)."""
    rng = np.random.default_rng(seed)
    total = int(np.prod(shape))
    nnz = max(1, int(total * density))
    flat = rng.choice(total, size=nnz, replace=False)
    flat.sort()
    coords = np.array(np.unravel_index(flat, shape), dtype=np.int64)
    data = rng.standard_normal(nnz).astype(np.float64)
    return coords, data, nnz


def mode0_unfold_indices(coords, shape):
    """N-мерные координаты row и col для mode-0"""
    rows = coords[0]
    cols = np.zeros_like(rows)
    n_rest = 1
    for k in range(len(shape) - 1, 0, -1):
        cols = cols + coords[k] * n_rest
        n_rest *= shape[k]
    return rows, cols, n_rest


def emit(text):
    print(text)
    REPORT_LINES.append(text)

## Конфигурации и проверка корректности

13 конфигов, покрывающих N = 2…8 и три порядка плотности.

In [ ]:
CONFIGS = [
    # 2D
    dict(shape=(1000, 1000),         density=1e-3),
    dict(shape=(1000, 1000),         density=1e-2),
    # 3D
    dict(shape=(100, 100, 100),      density=1e-3),
    dict(shape=(100, 100, 100),      density=1e-2),
    dict(shape=(200, 200, 200),      density=1e-4),
    dict(shape=(300, 300, 300),      density=1e-3),
    dict(shape=(300, 300, 300),      density=1e-2),
    # 4D
    dict(shape=(50, 50, 50, 50),     density=1e-3),
    dict(shape=(50, 50, 50, 50),     density=1e-2),
    # 5D–8D
    dict(shape=(20, 20, 20, 20, 20), density=1e-3),
    dict(shape=(10,) * 6,            density=1e-2),
    dict(shape=(8,)  * 7,            density=1e-2),
    dict(shape=(6,)  * 8,            density=1e-2),
]

K_TTM, J_TTM, K_ACCESS, SEED = 10, 50, 500, 0
FORMATS = ['CSR-N', 'pydata COO', 'pydata GCXS', 'scipy CSR*']

_c, _d, _ = make_random_tensor((30, 30, 30), 1e-2, seed=SEED)
_ref = sparse.COO(_c, _d, shape=(30, 30, 30)).todense()
_got = CSRN.from_coo(_c, _d, (30, 30, 30)).to_dense()
assert np.allclose(_ref, _got), 'CSR-N.to_dense расходится с pydata COO'

## Основной сетка построение / TTM0 / случайный доступ / память

In [ ]:
def run_one_config(shape, density, seed=SEED):
    N = len(shape)
    coords, data, nnz = make_random_tensor(shape, density, seed=seed)
    print(f'  shape={shape}, N={N}, nnz={nnz}, density={density:.0e}')

    # построение
    def build_csrn():     return CSRN.from_coo(coords, data, shape)
    def build_pds_coo():  return sparse.COO(coords, data, shape=shape)
    csrn_tensor = build_csrn()
    pds_coo     = build_pds_coo()
    def build_pds_gcxs(): return sparse.GCXS.from_coo(pds_coo)
    pds_gcxs    = build_pds_gcxs()
    def build_scipy():
        r, c, n = mode0_unfold_indices(coords, shape)
        return sp.csr_matrix((data, (r, c)), shape=(shape[0], n))
    scipy_csr_nd = build_scipy()

    t_build = {
        'CSR-N':       timeit(build_csrn),
        'pydata COO':  timeit(build_pds_coo),
        'pydata GCXS': timeit(build_pds_gcxs),
        'scipy CSR*':  timeit(build_scipy),
    }

    # TTM по моде 0
    rng_m = np.random.default_rng(seed + 1)
    matrices = [rng_m.standard_normal((J_TTM, shape[0])) for _ in range(K_TTM)]
    def run_csrn_ttm():
        for A in matrices: _ = csrn_tensor.ttm(A, 0)
    def run_coo_ttm():
        for A in matrices: _ = sparse.tensordot(A, pds_coo, axes=([1], [0]))
    def run_gcxs_ttm():
        for A in matrices: _ = sparse.tensordot(A, pds_gcxs, axes=([1], [0]))
    def run_scipy_ttm():
        for A in matrices: _ = A @ scipy_csr_nd
    t_ttm = {
        'CSR-N':       timeit(run_csrn_ttm),
        'pydata COO':  timeit(run_coo_ttm),
        'pydata GCXS': timeit(run_gcxs_ttm),
        'scipy CSR*':  timeit(run_scipy_ttm),
    }

    # случайный доступ
    rng_q = np.random.default_rng(seed + 2)
    queries = [tuple(rng_q.integers(0, s) for s in shape) for _ in range(K_ACCESS)]
    def run_csrn_access():
        for q in queries: _ = csrn_tensor[q]
    def run_coo_access():
        for q in queries: _ = pds_coo[q]
    def run_gcxs_access():
        for q in queries: _ = pds_gcxs[q]
    def run_scipy_access():
        for q in queries:
            col = 0
            for k in range(1, N):
                col = col * shape[k] + q[k]
            _ = scipy_csr_nd[q[0], col]
    t_access = {
        'CSR-N':       timeit(run_csrn_access),
        'pydata COO':  timeit(run_coo_access),
        'pydata GCXS': timeit(run_gcxs_access),
        'scipy CSR*':  timeit(run_scipy_access),
    }

    # память
    mem = {
        'CSR-N':       int(csrn_tensor.memory_bytes()),
        'pydata COO':  int(pds_coo.nbytes),
        'pydata GCXS': int(pds_gcxs.nbytes),
        'scipy CSR*':  int(scipy_csr_nd.data.nbytes + scipy_csr_nd.indices.nbytes + scipy_csr_nd.indptr.nbytes),
    }
    mem_dense = int(np.prod(shape)) * 8

    rows = []
    for fmt in FORMATS:
        rows.append({
            'shape': str(shape), 'N': N, 'density': density, 'nnz': nnz,
            'format': fmt,
            'build_s':       t_build[fmt],
            'ttm_s':         t_ttm[fmt],
            'access_s':      t_access[fmt],
            'memory_B':      mem[fmt],
            'mem_pct_dense': mem[fmt] / mem_dense * 100,
        })
    return rows


all_rows = []
for i, cfg in enumerate(CONFIGS, 1):
    print(f'[{i}/{len(CONFIGS)}] {cfg}')
    t0 = time.perf_counter()
    all_rows.extend(run_one_config(**cfg))
    print(f'    готово за {time.perf_counter() - t0:.1f} s')

df = pd.DataFrame(all_rows)
df['config'] = df.apply(
    lambda r: f"{r['shape']:<22} d={r['density']:.0e} nnz={r['nnz']:>6}",
    axis=1,
)
df.to_csv('results/grid.csv', index=False)
print(f'\nВсего строк: {len(df)} → results/grid.csv')

[1/13] {'shape': (1000, 1000), 'density': 0.001}
  shape=(1000, 1000), N=2, nnz=1000, density=1e-03
    готово за 14.6 s
[2/13] {'shape': (1000, 1000), 'density': 0.01}
  shape=(1000, 1000), N=2, nnz=10000, density=1e-02
    готово за 3.7 s
[3/13] {'shape': (100, 100, 100), 'density': 0.001}
  shape=(100, 100, 100), N=3, nnz=1000, density=1e-03
    готово за 7.6 s
[4/13] {'shape': (100, 100, 100), 'density': 0.01}
  shape=(100, 100, 100), N=3, nnz=10000, density=1e-02
    готово за 1.8 s
[5/13] {'shape': (200, 200, 200), 'density': 0.0001}
  shape=(200, 200, 200), N=3, nnz=800, density=1e-04
    готово за 1.6 s
[6/13] {'shape': (300, 300, 300), 'density': 0.001}
  shape=(300, 300, 300), N=3, nnz=27000, density=1e-03
    готово за 9.6 s
[7/13] {'shape': (300, 300, 300), 'density': 0.01}
  shape=(300, 300, 300), N=3, nnz=270000, density=1e-02
    готово за 39.6 s
[8/13] {'shape': (50, 50, 50, 50), 'density': 0.001}
  shape=(50, 50, 50, 50), N=4, nnz=6250, density=1e-03
    готово за 7.1 

## Итоговые таблицы по сетке

Четыретаблицы по метрикам + сводка ускорений CSR-N относительно бейзлайнов.

In [ ]:
for metric, label, fmt_fn in [
    ('build_s',  'Построение формата',                              fmt_time),
    ('ttm_s',    f'TTM по моде 0, K = {K_TTM} итераций',            fmt_time),
    ('access_s', f'Случайный доступ, {K_ACCESS} запросов',          fmt_time),
    ('memory_B', 'Память',                                          fmt_bytes),
]:
    pv = df.pivot(index='config', columns='format', values=metric).reindex(columns=FORMATS)
    pv = pv.apply(lambda col: col.map(fmt_fn))
    emit(f'\n### {label}\n')
    emit(tabulate(pv.reset_index(), headers='keys', tablefmt='github', showindex=False))

emit('\n### Выигрыши CSR-N относительно бейзлайнов (>1× — CSR-N лучше)\n')
rows_speed = []
for cfg_str, sub in df.groupby('config', sort=False):
    sub = sub.set_index('format')
    t_ttm_csrn    = sub.loc['CSR-N', 'ttm_s']
    t_access_csrn = sub.loc['CSR-N', 'access_s']
    mem_csrn      = sub.loc['CSR-N', 'memory_B']
    rows_speed.append({
        'config':        cfg_str,
        'TTM vs COO':    sub.loc['pydata COO',  'ttm_s']    / t_ttm_csrn,
        'TTM vs GCXS':   sub.loc['pydata GCXS', 'ttm_s']    / t_ttm_csrn,
        'TTM vs scipy*': sub.loc['scipy CSR*',  'ttm_s']    / t_ttm_csrn,
        'Acc vs COO':    sub.loc['pydata COO',  'access_s'] / t_access_csrn,
        'Acc vs GCXS':   sub.loc['pydata GCXS', 'access_s'] / t_access_csrn,
        'Mem vs COO':    sub.loc['pydata COO',  'memory_B'] / mem_csrn,
        'Mem vs GCXS':   sub.loc['pydata GCXS', 'memory_B'] / mem_csrn,
    })
speed_df = pd.DataFrame(rows_speed).set_index('config')
speed_df.to_csv('results/grid_speedup.csv')
speed_str = speed_df.apply(lambda col: col.map(lambda x: f'{x:5.2f}×')).reset_index()
emit(tabulate(speed_str, headers='keys', tablefmt='github', showindex=False))


### Построение формата

| config                                      | CSR-N     | pydata COO   | pydata GCXS   | scipy CSR*   |
|---------------------------------------------|-----------|--------------|---------------|--------------|
| (10, 10, 10, 10, 10, 10) d=1e-02 nnz= 10000 | 4.50 ms   | 480.3 µs     | 1.39 ms       | 379.6 µs     |
| (100, 100, 100)        d=1e-02 nnz= 10000   | 4.31 ms   | 543.1 µs     | 1.87 ms       | 447.9 µs     |
| (100, 100, 100)        d=1e-03 nnz=  1000   | 2.39 ms   | 104.1 µs     | 601.6 µs      | 299.6 µs     |
| (1000, 1000)           d=1e-02 nnz= 10000   | 3.15 ms   | 468.0 µs     | 3.92 ms       | 494.2 µs     |
| (1000, 1000)           d=1e-03 nnz=  1000   | 591.7 µs  | 138.8 µs     | 528.0 µs      | 216.9 µs     |
| (20, 20, 20, 20, 20)   d=1e-03 nnz=  3200   | 1.96 ms   | 291.3 µs     | 889.7 µs      | 339.3 µs     |
| (200, 200, 200)        d=1e-04 nnz=   800   | 382.8 µs  | 70.0 µs      | 358.7 µs      | 180.0 µs     |
| (300, 300, 300)    

## TTV: CSR-N vs pydata.sparse.tensordot

Свёртка с вектором по моде 0, нативная операция CSR-N.

In [ ]:
K_TTV = 10
ttv_rows = []
for cfg in CONFIGS:
    shape, density = cfg['shape'], cfg['density']
    coords, data, nnz = make_random_tensor(shape, density, seed=SEED)
    csrn = CSRN.from_coo(coords, data, shape)
    pds_coo = sparse.COO(coords, data, shape=shape)
    rng = np.random.default_rng(SEED + 50)
    vectors = [rng.standard_normal(shape[0]) for _ in range(K_TTV)]

    def run_csrn():
        for v in vectors: _ = csrn.ttv(v, 0)
    def run_pds():
        for v in vectors: _ = sparse.tensordot(v, pds_coo, axes=([0], [0]))

    t_csrn = timeit(run_csrn)
    t_pds  = timeit(run_pds)

    ttv_rows.append({
        'config':                       f'{shape} d={density:.0e} nnz={nnz}',
        f'CSR-N (×{K_TTV})':            fmt_time(t_csrn),
        f'pydata tensordot (×{K_TTV})': fmt_time(t_pds),
        'speedup':                      f'{t_pds / t_csrn:.2f}×',
    })

df_ttv = pd.DataFrame(ttv_rows)
df_ttv.to_csv('results/ttv.csv', index=False)
emit(f'\n### TTV: CSR-N vs pydata.sparse.tensordot (>1× — CSR-N быстрее)\n')
emit(tabulate(df_ttv, headers='keys', tablefmt='github', showindex=False))


### TTV: CSR-N vs pydata.sparse.tensordot (>1× — CSR-N быстрее)

| config                                     | CSR-N (×10)   | pydata tensordot (×10)   | speedup   |
|--------------------------------------------|---------------|--------------------------|-----------|
| (1000, 1000) d=1e-03 nnz=1000              | 669.8 µs      | 2.05 ms                  | 3.06×     |
| (1000, 1000) d=1e-02 nnz=10000             | 1.60 ms       | 6.01 ms                  | 3.76×     |
| (100, 100, 100) d=1e-03 nnz=1000           | 721.3 µs      | 2.10 ms                  | 2.91×     |
| (100, 100, 100) d=1e-02 nnz=10000          | 2.84 ms       | 6.17 ms                  | 2.17×     |
| (200, 200, 200) d=1e-04 nnz=800            | 876.9 µs      | 2.91 ms                  | 3.32×     |
| (300, 300, 300) d=1e-03 nnz=27000          | 11.61 ms      | 23.70 ms                 | 2.04×     |
| (300, 300, 300) d=1e-02 nnz=270000         | 142.79 ms     | 546.18 ms                | 3.83×     |
| (50, 50, 50, 5

## TTM с разреженной матрицей A

Внутреннее сравнение в CSR-N: метод `ttm(A, mode)` c автоматическим переключением на sparse-путь, если A `scipy.sparse`. При низкой плотности A sparse-путь даёт ускорение, при высокой dense выгоднее. Точка перехода где-то в районе 25–50% плотности A.

In [ ]:
A_DENSITIES = [0.01, 0.02, 0.05, 0.10, 0.15, 0.20, 0.30, 0.50, 1.0]
ttm_sparse_rows = []
for cfg in CONFIGS:
    shape, density = cfg['shape'], cfg['density']
    coords, data, nnz = make_random_tensor(shape, density, seed=SEED)
    csrn = CSRN.from_coo(coords, data, shape)
    rng = np.random.default_rng(SEED + 60)
    row = {'config': f'{shape} d={density:.0e} nnz={nnz}'}

    for d_A in A_DENSITIES:
        A_dense = rng.standard_normal((J_TTM, shape[0])) * (rng.random((J_TTM, shape[0])) < d_A)
        A_sparse = sp.csr_matrix(A_dense)

        def run_dense():  _ = csrn.ttm(A_dense, 0)
        def run_sparse(): _ = csrn.ttm(A_sparse, 0)

        t_dense  = timeit(run_dense)
        t_sparse = timeit(run_sparse)
        row[f'd_A={d_A}'] = f'{t_dense / t_sparse:.2f}×'

    ttm_sparse_rows.append(row)

df_ttm_sparse = pd.DataFrame(ttm_sparse_rows)
df_ttm_sparse.to_csv('results/ttm_sparse.csv', index=False)
emit('\n### Ускорение sparse-пути TTM относительно dense-пути (>1× — sparse выгоднее)\n')
emit(tabulate(df_ttm_sparse, headers='keys', tablefmt='github', showindex=False))


### Ускорение sparse-пути TTM относительно dense-пути (>1× — sparse выгоднее)

| config                                     | d_A=0.01   | d_A=0.02   | d_A=0.05   | d_A=0.1   | d_A=0.15   | d_A=0.2   | d_A=0.3   | d_A=0.5   | d_A=1.0   |
|--------------------------------------------|------------|------------|------------|-----------|------------|-----------|-----------|-----------|-----------|
| (1000, 1000) d=1e-03 nnz=1000              | 3.58×      | 5.54×      | 2.60×      | 1.95×     | 0.52×      | 2.50×     | 1.26×     | 0.60×     | 0.67×     |
| (1000, 1000) d=1e-02 nnz=10000             | 28.62×     | 17.19×     | 7.32×      | 4.73×     | 4.76×      | 5.79×     | 1.93×     | 1.26×     | 0.54×     |
| (100, 100, 100) d=1e-03 nnz=1000           | 2.83×      | 3.10×      | 2.76×      | 1.25×     | 3.21×      | 1.51×     | 1.35×     | 1.07×     | 0.64×     |
| (100, 100, 100) d=1e-02 nnz=10000          | 12.23×     | 7.28×      | 9.20×      | 4.80×     | 4.25×      | 2.12×     | 1.

## TTT

Y - `(shape[-1], 20)`. Бейзлайны: `sparse.tensordot` и densify+`np.tensordot`.

In [ ]:
ttt_rows = []
for cfg in CONFIGS:
    shape, density = cfg['shape'], cfg['density']
    N = len(shape)
    coords_x, data_x, nnz_x = make_random_tensor(shape, density, seed=SEED)
    shape_y = (shape[-1], 5)
    coords_y, data_y, nnz_y = make_random_tensor(shape_y, 0.1, seed=SEED + 70)

    X = CSRN.from_coo(coords_x, data_x, shape)
    Y = CSRN.from_coo(coords_y, data_y, shape_y)
    X_pds = sparse.COO(coords_x, data_x, shape=shape)
    Y_pds = sparse.COO(coords_y, data_y, shape=shape_y)

    mode_x, mode_y = N - 1, 0
    out_shape = shape[:-1] + (5,)
    dense_out_size = int(np.prod(out_shape))

    def run_csrn(): _ = X.ttt(Y, mode_x, mode_y)
    t_csrn = timeit(run_csrn)
    result = X.ttt(Y, mode_x, mode_y)
    nnz_out = result.nnz if hasattr(result, 'nnz') else 0

    def run_pds(): _ = sparse.tensordot(X_pds, Y_pds, axes=([mode_x], [mode_y]))
    t_pds = timeit(run_pds)

    t_ref = None
    try:
        X_dense, Y_dense = X.to_dense(), Y.to_dense()
        def run_ref(): _ = np.tensordot(X_dense, Y_dense, axes=([mode_x], [mode_y]))
        t_ref = timeit(run_ref)
        del X_dense, Y_dense
    except MemoryError:
        pass

    fill = nnz_out / dense_out_size if dense_out_size > 0 else 0
    ttt_rows.append({
        'X shape':    f'{shape} nnz={nnz_x}',
        'Y shape':    f'{shape_y} nnz={nnz_y}',
        'CSR-N':      fmt_time(t_csrn),
        'pydata':     fmt_time(t_pds),
        'densify':    fmt_time(t_ref) if t_ref else 'OOM',
        'vs pydata':  f'{t_pds / t_csrn:.2f}×',
        'vs densify': f'{t_ref / t_csrn:.2f}×' if t_ref else '—',
        'fill_out':   f'{fill*100:.2f}%',
    })

df_ttt = pd.DataFrame(ttt_rows)
df_ttt.to_csv('results/ttt.csv', index=False)
emit('\n### TTT: CSR-N vs pydata vs densify (>1× — CSR-N быстрее)\n')
emit(tabulate(df_ttt, headers='keys', tablefmt='github', showindex=False))


### TTT: CSR-N vs pydata vs densify (>1× — CSR-N быстрее)

| X shape                            | Y shape           | CSR-N     | pydata   | densify   | vs pydata   | vs densify   | fill_out   |
|------------------------------------|-------------------|-----------|----------|-----------|-------------|--------------|------------|
| (1000, 1000) nnz=1000              | (1000, 5) nnz=500 | 701.5 µs  | 715.3 µs | 1.37 ms   | 1.02×       | 1.95×        | 9.32%      |
| (1000, 1000) nnz=10000             | (1000, 5) nnz=500 | 3.79 ms   | 2.38 ms  | 1.61 ms   | 0.63×       | 0.42×        | 62.12%     |
| (100, 100, 100) nnz=1000           | (100, 5) nnz=50   | 1.10 ms   | 1.48 ms  | 1.83 ms   | 1.34×       | 1.66×        | 1.04%      |
| (100, 100, 100) nnz=10000          | (100, 5) nnz=50   | 4.91 ms   | 3.61 ms  | 1.63 ms   | 0.74×       | 0.33×        | 9.38%      |
| (200, 200, 200) nnz=800            | (200, 5) nnz=100  | 843.6 µs  | 2.23 ms  | 12.56 ms  | 2.64×       | 14.89×       | 0

## Сохранение отчёта

In [ ]:
with open('results/report.md', 'w') as f:
    f.write('# Результаты бенчмарка CSR-N\n')
    f.write('\n'.join(REPORT_LINES))
    f.write('\n')

print('Сохранено:')
for p in sorted(os.listdir('results')):
    sz = os.path.getsize(os.path.join('results', p))
    print(f'  results/{p}  ({sz} B)')

Сохранено:
  results/grid.csv  (8837 B)
  results/grid_speedup.csv  (2392 B)
  results/report.md  (15542 B)
  results/ttm_sparse.csv  (1398 B)
  results/ttt.csv  (1278 B)
  results/ttv.csv  (863 B)
